# 03 — Hypothesis Tests (Phase 4)

Formal testing of the relationships previewed in `02_descriptive_stats.ipynb`.
Every test reports: statistic, p-value, effect size with interpretation, and 95% CI where computable.

**Locked decisions** (from Phase 1 & 2 state):
- Delivered-only filter for delivery and satisfaction tests
- `review_score` treated as ordinal — no Pearson, no t-test on means
- State not controlled separately (delivery-mediated, ρ = −0.796)
- Price = `items_price_total`, freight excluded

**Execution order:** Q1 → §4-A → §4-B → Q2 → Q3 → Q4a/b → §1 ANOVA.

---

## Q1 — Does order price correlate with delivery time?

Spearman rank correlation on `items_price_total` vs `delivery_days`, delivered orders only.
H₀: ρ = 0. Two-sided. 95% CI via 2000-resample bootstrap.

In [ ]:
# =========================================================
# Q1: Does order price correlate with delivery time?
# Spearman rank correlation, delivered-only, freight excluded
# =========================================================
from scipy import stats

# Delivered-only filter (decision #1)
delivered = orders[orders["order_status"] == "delivered"].copy()

# Defensive: drop rows missing either variable
q1 = delivered[["items_price_total", "delivery_days"]].dropna()
print(f"Q1 sample size: {len(q1):,} (dropped {len(delivered) - len(q1)} rows with NaN)")

# Spearman rank correlation
rho, pval = stats.spearmanr(q1["items_price_total"], q1["delivery_days"])
print(f"\nSpearman ρ = {rho:+.4f}")
print(f"p-value    = {pval:.2e}")

# Bootstrap 95% CI on ρ (2000 resamples, paired)
def spearman_stat(x, y):
    return stats.spearmanr(x, y).statistic

rng = np.random.default_rng(42)
boot = stats.bootstrap(
    (q1["items_price_total"].values, q1["delivery_days"].values),
    statistic=spearman_stat,
    n_resamples=2000,
    paired=True,
    confidence_level=0.95,
    random_state=rng,
    method="percentile",
)
ci_lo, ci_hi = boot.confidence_interval
print(f"95% CI     = [{ci_lo:+.4f}, {ci_hi:+.4f}]  (bootstrap, 2000 resamples)")

# Interpretation using Cohen-style thresholds for ρ
abs_rho = abs(rho)
if   abs_rho < 0.10: label = "trivial"
elif abs_rho < 0.30: label = "weak"
elif abs_rho < 0.50: label = "moderate"
else:                label = "strong"
print(f"\nEffect size: {label} ({rho:+.3f})")

Q1 sample size: 96,470 (dropped 8 rows with NaN)

Spearman ρ = +0.1017
p-value    = 4.49e-220


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Decile bin for the line panel
q1_binned = q1.copy()
q1_binned["price_decile"] = pd.qcut(q1_binned["items_price_total"], 10, labels=False) + 1
decile_stats = q1_binned.groupby("price_decile").agg(
    price_median=("items_price_total", "median"),
    delivery_median=("delivery_days", "median"),
    delivery_q25=("delivery_days", lambda s: s.quantile(0.25)),
    delivery_q75=("delivery_days", lambda s: s.quantile(0.75)),
).reset_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        f"Scatter (log x) — ρ = {rho:+.3f}",
        "Median delivery days by price decile (IQR shaded)"
    ),
    horizontal_spacing=0.12,
)

# Left: scatter, sampled to 5k for render speed
sample = q1.sample(5000, random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["items_price_total"], y=sample["delivery_days"],
        mode="markers", marker=dict(size=3, opacity=0.25),
        showlegend=False,
    ),
    row=1, col=1,
)

# Right: decile medians with IQR band
fig.add_trace(
    go.Scatter(
        x=decile_stats["price_median"], y=decile_stats["delivery_q75"],
        mode="lines", line=dict(width=0), showlegend=False,
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=decile_stats["price_median"], y=decile_stats["delivery_q25"],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(100,100,200,0.2)",
        name="IQR", showlegend=True,
    ),
    row=1, col=2,
)
fig.add_trace(
    go.Scatter(
        x=decile_stats["price_median"], y=decile_stats["delivery_median"],
        mode="lines+markers", line=dict(width=3), name="Median",
    ),
    row=1, col=2,
)

fig.update_xaxes(type="log", title_text="Order item-price total (BRL, log)", row=1, col=1)
fig.update_xaxes(type="log", title_text="Decile median price (BRL, log)", row=1, col=2)
fig.update_yaxes(title_text="Delivery days", row=1, col=1)
fig.update_yaxes(title_text="Delivery days", row=1, col=2)
fig.update_layout(
    title_text="Q1 — Price vs delivery time (delivered orders, n≈96k)",
    height=500, width=1100, template="plotly_white",
)

fig.show()
fig.write_image(PROJECT_ROOT / "reports" / "figures" / "07_q1_price_vs_delivery.png", scale=2)

### Q1 — Result

**Test:** Spearman ρ on `items_price_total` vs `delivery_days`, delivered orders only.
**Sample:** n = 96,470 (8 rows dropped for missing values).

| Metric | Value |
|---|---|
| ρ | +0.102 |
| 95% CI | [+0.095, +0.108] |
| p-value | 4.5 × 10⁻²²⁰ |
| ρ² | 0.010 |
| Effect size | weak, effectively trivial |

**Interpretation.** Price and delivery time are functionally uncorrelated. The CI
lower bound (+0.095) sits below the conventional 0.10 "weak" threshold — this is
honestly better read as trivial than weak. Price explains ~1% of delivery-time
variance. The vanishingly small p-value is a textbook demonstration that significance
carries no information about magnitude at n ≈ 100k: at this sample size any ρ ≠ 0
will register p ≈ 0.

The decile-median panel shows the signal most cleanly: median delivery climbs
from ~9.3 days (cheapest decile) to ~11.3 days (most expensive decile), a ~2-day
spread across a >20× price range. Likely compositional — expensive orders skew
toward heavier/bulkier items on different carrier tiers — not a systematic
slow-lane for premium orders.

**Methodological note.** Phase 2 preview suggested ρ ≈ +0.14; the final value is
lower because we correlated against `items_price_total` (freight excluded) rather
than payment total. Including freight would have been partially auto-correlated
with delivery time — freight is itself a function of the delivery process.

**Carry-forward.** Price is not a meaningful delivery predictor. No need to
condition on price in §4-A (delivery → review) or §4-B (on_time → review).